In [0]:
%pip install openpyxl

In [0]:
import json, re, os
import pandas as pd
from openpyxl import load_workbook

SRC = "/Volumes/energy/ops/raw_source"
OUT = "/Volumes/energy/bronze/landing/grid"
os.makedirs(OUT, exist_ok=True)

#ONLY = ["September 2021.xlsx", "October 2022.xlsx", "January 2024- June 2025.xlsx"]  # test run; set to None for all
ONLY = None
OVERWRITE = False

MONTHS = {m: i for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], start=1)}

SCADA_COLS = {
    "TOTAL_THM_ONLY|P": "thermal_mw",
    "TOTAL_NLDC_HYD|P": "hydro_mw",
    "TOTAL_NLDC_GAS|P": "gas_mw",
    "TOTAL_NLDC_NPC|P": "nuclear_mw",
    "ALL_INDIA_WIND|P": "wind_mw",
    "ALL_IND_SOLAR|P": "solar_mw",
    "NLDC_DEMAND|P": "demand_mw",
}
REPORT_COLS = {
    "Timestamp": "ts_raw",
    "Demand (MW)": "demand_mw",
    "Wind (MW)": "wind_mw",
    "Solar (MW)": "solar_mw",
    "Total Generation (MW)": "total_generation_mw",
}
OUT_COLS = ["ts_raw", "thermal_mw", "hydro_mw", "gas_mw", "nuclear_mw",
            "wind_mw", "solar_mw", "demand_mw", "total_generation_mw"]


def output_name(fname):
    """'Novemeber 2023.xlsx' -> 'grid_2023-11.csv' (first 3 letters survive the typo)."""
    years = re.findall(r"\d{4}", fname)
    if len(years) > 1:                                   # the multi-month file
        return f"grid_{years[0]}-01_to_{years[-1]}-06.csv"
    return f"grid_{years[0]}-{MONTHS[fname.lower()[:3]]:02d}.csv"


def read_scada(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    cfg_json = list(wb["_osi_config"].iter_rows(min_row=2, max_row=2, values_only=True))[0][0]
    wb.close()
    resolution = int(json.loads(cfg_json)["r"])

    df = pd.read_excel(path, sheet_name="Sheet1", header=1, engine="openpyxl")
    df = df.rename(columns={df.columns[0]: "ts_raw", **SCADA_COLS})
    df["ts_raw"] = pd.to_datetime(df["ts_raw"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    return df, "scada", resolution


def read_report(path):
    df = pd.read_excel(path, sheet_name="Report", engine="openpyxl", dtype={"Timestamp": str})
    return df.rename(columns=REPORT_COLS), "report", 3600

In [0]:
files = sorted(f.name for f in dbutils.fs.ls(SRC) if f.name.endswith(".xlsx"))
if ONLY:
    files = [f for f in files if f in ONLY]

log = []
for fname in files:
    target = f"{OUT}/{output_name(fname)}"
    if os.path.exists(target) and not OVERWRITE:
        print("skip (exists)", fname)
        continue

    path = f"{SRC}/{fname}"
    sheets = load_workbook(path, read_only=True).sheetnames
    df, fmt, res = read_report(path) if "Report" in sheets else read_scada(path)

    for c in OUT_COLS:
        if c not in df.columns:
            df[c] = None
    df = df[OUT_COLS]
    df["source_file"] = fname
    df["source_format"] = fmt
    df["resolution_seconds"] = res

    df.to_csv(target, index=False)
    log.append((fname, os.path.basename(target), fmt, res, len(df),
                str(df["ts_raw"].iloc[0]), str(df["ts_raw"].iloc[-1])))
    print(f"{fname:32s} -> {os.path.basename(target):32s} {fmt:6s} {res:>5}s {len(df):>9,} rows")

In [0]:
if log:
    (spark.createDataFrame(log, "source_file STRING, output_file STRING, source_format STRING, "
                                "resolution_seconds INT, rows LONG, first_ts STRING, last_ts STRING")
          .write.mode("append").saveAsTable("energy.ops.extract_log"))

display(spark.table("energy.ops.extract_log").orderBy("output_file"))

In [0]:
dbutils.fs.mkdirs("/Volumes/energy/bronze/landing/state_daily")
dbutils.fs.cp(f"{SRC}/long_data_.csv",
              "/Volumes/energy/bronze/landing/state_daily/state_daily_2019_2020.csv")

In [0]:
c = pd.read_csv(f"{OUT}/grid_2024-01_to_2025-06.csv")
print(c.tail(6).to_string())

bad = c[~c["ts_raw"].astype(str).str.match(r"^\d{2}-\d{2}-\d{4} \d{2}:\d{2}:\d{2}$")]
print(f"\nRows without a valid timestamp: {len(bad)}")
print(bad.to_string())

In [0]:
d = c.iloc[:-6].copy()                       # drop the 6 summary rows for this check
ts = pd.to_datetime(d["ts_raw"], format="%d-%m-%Y %H:%M:%S")
off = (ts.dt.minute != 0) | (ts.dt.second != 0)

print("off-the-hour rows: ", off.sum())
print("duplicate timestamps:", ts.duplicated().sum())
print("zero-demand rows:  ", (d["demand_mw"].astype(float) == 0).sum())
print("demand > 250 GW:   ", (d["demand_mw"].astype(float) > 250_000).sum())
print(d[off].head(10).to_string())

In [0]:
dbutils.fs.mkdirs("/Volumes/energy/bronze/landing/state_monthly")
dbutils.fs.cp("/Volumes/energy/ops/raw_source/energy_req_avail_monthly.csv",
              "/Volumes/energy/bronze/landing/state_monthly/energy_req_avail_monthly.csv")